In [29]:
import pandas as pd
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, cross_val_score
import mlflow
from mlflow.models import infer_signature


In [30]:
## set the tracking uri
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

In [31]:
## load the dataset
X, y = datasets.load_iris(return_X_y=True)

# split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20)

# Define the model hyperparameters
params = {"penalty":"l2","solver": "lbfgs", "max_iter": 1000, "multi_class": "auto", "random_state": 8888}

##train the model
lr=LogisticRegression(**params)
lr.fit(X_train,y_train)

c:\Users\Jose Mendoza\OneDrive\Escritorio\Projects\mlops-deployment-demo\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,8888
,solver,'lbfgs'
,max_iter,1000
,multi_class,'auto'


In [32]:
## Prediction on the test set
y_pred = lr.predict(X_test)
y_pred

array([2, 1, 2, 0, 2, 1, 1, 2, 1, 0, 1, 1, 0, 2, 1, 2, 0, 1, 2, 0, 1, 2,
       2, 0, 1, 0, 0, 1, 0, 1])

In [33]:
# 🧮 Evaluate on the hold-out test set
accuracy = accuracy_score(y_test, y_pred)
print("Hold-out test accuracy:", accuracy)

# 🔁 Run 5-fold cross-validation on full dataset
cv_scores = cross_val_score(lr, X, y, cv=5)
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()
print(f"Cross-validated accuracy (mean ± std): {cv_mean:.3f} ± {cv_std:.3f}")

Hold-out test accuracy: 0.9333333333333333
Cross-validated accuracy (mean ± std): 0.973 ± 0.025


c:\Users\Jose Mendoza\OneDrive\Escritorio\Projects\mlops-deployment-demo\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Jose Mendoza\OneDrive\Escritorio\Projects\mlops-deployment-demo\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Jose Mendoza\OneDrive\Escritorio\Projects\mlops-deployment-demo\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this war

In [34]:
### MLFLOW tracking 
# This sets the URL where MLflow logs and UI are running.
# In this case, it's pointing to your local MLflow Tracking Server.
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

## Create a new MLFLOW experiment
# Think of an experiment as a folder that contains multiple runs.
# You can group and compare all your training runs under this experiment name.
mlflow.set_experiment("MLFLOW Quickstart")

## Start an MLFLOW run
# This opens a logging context: everything inside (params, metrics, models)
# will be tracked and saved in MLflow for this specific run.
with mlflow.start_run():

    ## Log the hyperparameters
    # Logs all model configuration values (like solver, max_iter, etc.)
    # so you can later reproduce and compare runs.
    mlflow.log_params(params)
    
    ## Log evaluation metrics
    # Logs both hold-out accuracy and cross-validated accuracy
    # so you can monitor improvements across runs.
    mlflow.log_metric("holdout_accuracy", accuracy)
    mlflow.log_metric("cv_accuracy_mean", cv_mean)
    mlflow.log_metric("cv_accuracy_std", cv_std)

    ## Set a tag with additional metadata
    # Tags are useful for annotating runs (e.g., purpose, model type, notes)
    mlflow.set_tag("Training Info", "Basic LR model for iris data")

    ## Infer the model signature
    # This captures input/output schema — helps when serving the model later.
    signature = infer_signature(X_train, lr.predict(X_train))
    
    ## Log the model
    # Saves the trained model in MLflow format.
    # Also registers the model (optional) with a version-controlled name.
    # You’ll be able to track it, download it, and deploy it later.
    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        artifact_path="iris_model",  # Folder name inside the run
        signature=signature,
        input_example=X_train,       # Optional: helps document usage
        registered_model_name="tracking-quickstart"  # Will appear in the Model Registry
    )


2025/08/03 12:51:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'tracking-quickstart' already exists. Creating a new version of this model...
2025/08/03 12:51:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 5


🏃 View run zealous-mare-570 at: http://127.0.0.1:5000/#/experiments/521920103584845037/runs/35b6ccd984274f0c9f503dc4e24d2630
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/521920103584845037


Created version '5' of model 'tracking-quickstart'.


## 📦 MLflow Key Concepts Summary

### 🧠 Core Terms/ Priority (from most critical to optional)

| Priority  | Term                        | What it does                                                         | Why?                                                                              |
| --------- | --------------------------- | -------------------------------------------------------------------- | --------------------------------------------------------------------------------- |
| 🥇 Must   | **`set_experiment()`**      | Groups related runs under a named experiment (like a folder of runs) | Organises your runs; needed to track properly                                     |
| 🥈 High   | **`artifact_path`**         | Where the **model files are saved** *inside* the run (a subfolder)   | Required for saving the model                                                     |
| 🥉 Medium | **`registered_model_name`** | Adds the model to the **Model Registry** with a name and versioning  | Optional — but needed if you want **versioning** or **staging/production** models |
| 🧾 Nice   | **`set_tag()`**             | Adds custom metadata to a run (notes, model type, comments, etc.)    | Optional — for **human readability** (notes, model type, etc.)                    |


---

## 📁 Visual Diagram: How MLflow Organises Artifacts

Here’s a simplified diagram of how everything fits together when you log a model:

```
MLflow Tracking Server
│
└── Experiments/
    └── MLFLOW Quickstart (set_experiment)
        └── Run ID: abc123/
            ├── tags/                 ← set_tag (key-value metadata)
            ├── params/               ← log_params (hyperparameters)
            ├── metrics/              ← log_metric (accuracy, loss, etc.)
            ├── artifacts/
            │   └── iris_model/       ← artifact_path ("iris_model")
            │       ├── MLmodel       ← metadata + model format
            │       └── model.pkl     ← saved model file
            └── model registry/       ← (linked if registered_model_name is set)
```

---

In [35]:
# 🔁 This is your second model in the same notebook
# We're testing a different solver: 'newton-cg'

from sklearn.linear_model import LogisticRegression

# 🧪 Define new model parameters
params_newton = {
    "solver": "newton-cg",     # Different solver
    "max_iter": 1000,
    "multi_class": "auto",
    "random_state": 42
}

# ⚙️ Train new model with newton-cg
lr_newton = LogisticRegression(**params_newton)
lr_newton.fit(X_train, y_train)

# 📈 Predict and evaluate
y_pred_newton = lr_newton.predict(X_test)
accuracy_newton = accuracy_score(y_test, y_pred_newton)

# 🔁 Run 5-fold cross-validation using full dataset
cv_scores = cross_val_score(lr_newton, X, y, cv=5)
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

# 🖨️ Output both metrics
print("Hold-out test accuracy:", accuracy_newton)
print("Cross-validated accuracy (mean ± std):", f"{cv_mean:.3f} ± {cv_std:.3f}")

# 🚀 Log this second model in the same experiment
# (you already called set_tracking_uri and set_experiment earlier)

with mlflow.start_run():  # New run for the second model

    # 📝 Log the second model's hyperparameters
    mlflow.log_params(params_newton)

    # 📊 Log both evaluation methods
    mlflow.log_metric("holdout_accuracy", accuracy_newton)
    mlflow.log_metric("cv_accuracy_mean", cv_mean)
    mlflow.log_metric("cv_accuracy_std", cv_std)

    # 🏷️ Add tags for traceability
    mlflow.set_tag("Solver", "newton-cg")
    mlflow.set_tag("Note", "Second model run using newton-cg")

    # 🔍 Infer input/output schema
    signature_newton = infer_signature(X_train, lr_newton.predict(X_train))

    # 💾 Save this model under a different artifact path and name
    model_info_newton = mlflow.sklearn.log_model(
        sk_model=lr_newton,
        artifact_path="iris_model_newtoncg",           # Different folder
        signature=signature_newton,
        input_example=X_train,
        registered_model_name="iris-lr-newtoncg"       # Unique model name
    )


c:\Users\Jose Mendoza\OneDrive\Escritorio\Projects\mlops-deployment-demo\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Jose Mendoza\OneDrive\Escritorio\Projects\mlops-deployment-demo\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Jose Mendoza\OneDrive\Escritorio\Projects\mlops-deployment-demo\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this war

Hold-out test accuracy: 0.9333333333333333
Cross-validated accuracy (mean ± std): 0.973 ± 0.025


2025/08/03 12:51:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'iris-lr-newtoncg' already exists. Creating a new version of this model...
2025/08/03 12:51:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-lr-newtoncg, version 4


🏃 View run gifted-lamb-327 at: http://127.0.0.1:5000/#/experiments/521920103584845037/runs/151e3293cc504c6cb31c57a2d1c4ac99
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/521920103584845037


Created version '4' of model 'iris-lr-newtoncg'.


## 🔍 Model Validation and Inference Using MLflow (PyFunc + Signature Check)

This cell demonstrates how to:

1. ✅ **Validate incoming payloads** before serving a model, using `mlflow.models.validate_serving_input`.  
2. 📥 **Load a saved model** from a previous MLflow run using `pyfunc.load_model`.  
3. 🔮 **Make predictions** on test data (`X_test`) using the loaded model.  
4. 📊 **Compare predictions vs actual values** in a well-formatted DataFrame for inspection or debugging.

This process helps ensure that the model is both deployable and still performing correctly after serialization.

In [43]:
from mlflow.models import validate_serving_input
import json

# 📦 Get the model URI that MLflow uses to locate the saved model
# This URI comes from a previous MLflow run where the model was logged
model_uri = model_info.model_uri  # You can also use model_info_newton.model_uri if it's your second model

# 🧪 Simulate a real-world prediction request by defining a JSON-like payload
# This is what a REST API would send to your model after deployment
# Each inner list is a sample with 4 features (just like the iris dataset)
serving_payload = """
{
  "inputs": [
    [5.7, 3.8, 1.7, 0.3],
    [6.3, 2.7, 4.9, 1.8]
  ]
}
"""

# ✅ Check if the provided inputs match the model's expected input signature
# This will raise an error if the payload structure is incorrect
validate_serving_input(model_uri, serving_payload)
print("✅ Input is valid and matches the model signature!")

# 📥 Load the model from MLflow’s storage location using the PyFunc interface
# This allows you to treat the model as a standard Python function
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

# 🔮 Make predictions on the real test set (X_test) using the loaded model
# This is useful to confirm that the saved model works after loading
predictions = loaded_model.predict(X_test)

# 🧾 Get the column names for the iris dataset to format the DataFrame nicely
iris_features_name = datasets.load_iris().feature_names

# 🧪 Create a DataFrame to compare model predictions with actual test labels
# This is helpful for debugging, reporting, or inspecting prediction quality
result = pd.DataFrame(X_test, columns=iris_features_name)
result["actual_class"] = y_test                    # True labels
result["predicted_class"] = predictions            # Model predictions


✅ Input is valid and matches the model signature!
